# Setup + Find Original Dataset


In [2]:
!pip install -q ultralytics huggingface_hub scikit-learn tqdm

from pathlib import Path
import shutil
import yaml
import pandas as pd
import numpy as np
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")

# Find the original 4-class YOLO dataset
candidate_dirs = []

for yaml_path in INPUT_ROOT.rglob("data.yaml"):
    folder = yaml_path.parent
    
    if not (folder / "images").exists():
        continue
    
    if not (folder / "labels").exists():
        continue
    
    try:
        cfg = yaml.safe_load(yaml_path.read_text())
        names = cfg.get("names", [])
        
        if isinstance(names, dict):
            names_list = list(names.values())
        else:
            names_list = list(names)
        
        score = 0
        joined_names = " ".join([str(x).lower() for x in names_list])
        
        if "player_team1" in joined_names:
            score += 10
        if "player_team2" in joined_names:
            score += 10
        if "ball" in joined_names:
            score += 5
        if "referee" in joined_names:
            score += 5
        
        candidate_dirs.append((score, folder, names_list))
        
    except Exception:
        candidate_dirs.append((0, folder, []))

if len(candidate_dirs) == 0:
    raise FileNotFoundError("Could not find YOLO dataset in /kaggle/input")

candidate_dirs = sorted(candidate_dirs, key=lambda x: x[0], reverse=True)
original_dataset_dir = candidate_dirs[0][1]

print("Original dataset:", original_dataset_dir)
print("Original names:", candidate_dirs[0][2])

for score, folder, names in candidate_dirs:
    print(score, folder, names)

Torch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
Original dataset: /kaggle/input/datasets/hashemalshaweesh/hamzasplit1/football_yolo_dataset/football_yolo_dataset
Original names: ['player_team1', 'player_team2', 'ball', 'referee']
30 /kaggle/input/datasets/hashemalshaweesh/hamzasplit1/football_yolo_dataset/football_yolo_dataset ['player_team1', 'player_team2', 'ball', 'referee']


# Convert Dataset to 3 Classes + Oversampling

In [3]:
from pathlib import Path
import shutil
import yaml
import pandas as pd

converted_dataset_dir = Path("/kaggle/working/football_yolo_dataset_3cls_improved")

if converted_dataset_dir.exists():
    shutil.rmtree(converted_dataset_dir)

# Old classes:
# 0 player_team1 -> 0 player
# 1 player_team2 -> 0 player
# 2 ball         -> 1 ball
# 3 referee      -> 2 referee

class_mapping = {
    0: 0,
    1: 0,
    2: 1,
    3: 2,
}

new_class_names = ["player", "ball", "referee"]

rows = []

for split in ["train", "val", "test"]:
    src_img_dir = original_dataset_dir / "images" / split
    src_lbl_dir = original_dataset_dir / "labels" / split

    dst_img_dir = converted_dataset_dir / "images" / split
    dst_lbl_dir = converted_dataset_dir / "labels" / split

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    for img_path in sorted(src_img_dir.glob("*.jpg")):
        shutil.copy2(img_path, dst_img_dir / img_path.name)

        src_label_path = src_lbl_dir / f"{img_path.stem}.txt"
        dst_label_path = dst_lbl_dir / f"{img_path.stem}.txt"

        new_lines = []

        if src_label_path.exists():
            for line in src_label_path.read_text().splitlines():
                parts = line.strip().split()

                if len(parts) != 5:
                    continue

                old_cls = int(float(parts[0]))

                if old_cls not in class_mapping:
                    continue

                new_cls = class_mapping[old_cls]
                new_lines.append(" ".join([str(new_cls)] + parts[1:]))

                rows.append({
                    "split": split,
                    "class_id": new_cls,
                    "class_name": new_class_names[new_cls]
                })

        dst_label_path.write_text("\n".join(new_lines) + ("\n" if new_lines else ""))

stats_df = pd.DataFrame(rows)

# Create weighted training list to help ball and referee
train_images_dir = converted_dataset_dir / "images" / "train"
train_labels_dir = converted_dataset_dir / "labels" / "train"

weighted_train_txt = converted_dataset_dir / "train_weighted.txt"
weighted_lines = []

for img_path in sorted(train_images_dir.glob("*.jpg")):
    label_path = train_labels_dir / f"{img_path.stem}.txt"

    has_ball = False
    has_referee = False

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            cls_id = int(float(parts[0]))

            if cls_id == 1:
                has_ball = True

            if cls_id == 2:
                has_referee = True

    repeat = 1

    # Ball is small and important
    if has_ball:
        repeat += 1

    # Referee was weak, so give referee frames more weight
    if has_referee:
        repeat += 2

    for _ in range(repeat):
        weighted_lines.append(str(img_path))

weighted_train_txt.write_text("\n".join(weighted_lines))

data_yaml_3cls = converted_dataset_dir / "data.yaml"

data_config = {
    "path": str(converted_dataset_dir),
    "train": str(weighted_train_txt),
    "val": "images/val",
    "test": "images/test",
    "nc": 3,
    "names": new_class_names,
}

with open(data_yaml_3cls, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print("Converted dataset:", converted_dataset_dir)
print("\ndata.yaml:")
print(data_yaml_3cls.read_text())

print("\nImage counts:")
for split in ["train", "val", "test"]:
    n_images = len(list((converted_dataset_dir / "images" / split).glob("*.jpg")))
    n_labels = len(list((converted_dataset_dir / "labels" / split).glob("*.txt")))
    print(f"{split}: {n_images} images | {n_labels} labels")

print("\nClass instances:")
display(stats_df.groupby(["split", "class_name"]).size().reset_index(name="instances"))

print("\nWeighted train images:", len(weighted_lines))

Converted dataset: /kaggle/working/football_yolo_dataset_3cls_improved

data.yaml:
path: /kaggle/working/football_yolo_dataset_3cls_improved
train: /kaggle/working/football_yolo_dataset_3cls_improved/train_weighted.txt
val: images/val
test: images/test
nc: 3
names:
- player
- ball
- referee


Image counts:
train: 4280 images | 4280 labels
val: 1462 images | 1462 labels
test: 720 images | 720 labels

Class instances:


,split,class_name,instances
0,test,ball,626
1,test,player,4702
2,test,referee,518
3,train,ball,3557
4,train,player,44813
5,train,referee,5194
6,val,ball,1414
7,val,player,16137
8,val,referee,1614



Weighted train images: 14016


# Train Improved 3-Class Model

In [4]:
from ultralytics import YOLO
from huggingface_hub import hf_hub_download
from pathlib import Path
import pandas as pd
import contextlib
import time
import torch
import gc

PROJECT_DIR = Path("/kaggle/working/three_class_improved_experiments")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

hf_model_path = hf_hub_download(
    repo_id="uisikdag/yolo-v8-football-players-detection",
    filename="best.pt"
)

exp_name = "hf_football_finetuned_3cls_improved_960"

start_time = time.time()

model = YOLO(hf_model_path)

train_log_path = PROJECT_DIR / f"{exp_name}_train_log.txt"

with open(train_log_path, "w") as log_file:
    with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
        model.train(
            data=str(data_yaml_3cls),
            epochs=80,
            imgsz=960,
            batch=8,
            device=0,
            project=str(PROJECT_DIR),
            name=exp_name,
            patience=20,
            plots=True,
            val=True,
            exist_ok=True,
            verbose=False,

            optimizer="AdamW",
            lr0=0.001,
            lrf=0.01,
            weight_decay=0.0005,
            warmup_epochs=3,
            cos_lr=True,
            amp=True,

            hsv_h=0.015,
            hsv_s=0.70,
            hsv_v=0.40,
            degrees=2.0,
            translate=0.10,
            scale=0.50,
            fliplr=0.50,
            mosaic=0.50,
            mixup=0.05,
            close_mosaic=15,

            workers=2,
            cache=False,
            save_period=10
        )

best_model_path = PROJECT_DIR / exp_name / "weights" / "best.pt"

if not best_model_path.exists():
    raise FileNotFoundError(f"Best model not found: {best_model_path}")

best_model = YOLO(str(best_model_path))

test_log_path = PROJECT_DIR / f"{exp_name}_test_log.txt"

with open(test_log_path, "w") as log_file:
    with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
        test_metrics = best_model.val(
            data=str(data_yaml_3cls),
            split="test",
            imgsz=960,
            batch=8,
            device=0,
            project=str(PROJECT_DIR),
            name=f"{exp_name}_test",
            plots=True,
            exist_ok=True,
            verbose=False
        )

elapsed_minutes = round((time.time() - start_time) / 60, 2)

summary_df = pd.DataFrame([{
    "experiment": exp_name,
    "imgsz": 960,
    "batch": 8,
    "epochs": 80,
    "patience": 20,
    "time_min": elapsed_minutes,
    "precision": round(float(test_metrics.results_dict["metrics/precision(B)"]), 4),
    "recall": round(float(test_metrics.results_dict["metrics/recall(B)"]), 4),
    "mAP50": round(float(test_metrics.results_dict["metrics/mAP50(B)"]), 4),
    "mAP50_95": round(float(test_metrics.results_dict["metrics/mAP50-95(B)"]), 4),
    "best_model_path": str(best_model_path),
}])

class_names = ["player", "ball", "referee"]
box = test_metrics.box

per_class_rows = []

for i, class_name in enumerate(class_names):
    per_class_rows.append({
        "experiment": exp_name,
        "class": class_name,
        "precision": round(float(box.p[i]), 4),
        "recall": round(float(box.r[i]), 4),
        "mAP50": round(float(box.ap50[i]), 4),
        "mAP50_95": round(float(box.ap[i]), 4),
    })

per_class_df = pd.DataFrame(per_class_rows)

summary_df.to_csv("/kaggle/working/three_class_improved_summary.csv", index=False)
per_class_df.to_csv("/kaggle/working/three_class_improved_per_class.csv", index=False)

BEST_MODEL_PATH = str(best_model_path)

print("Overall results:")
display(summary_df)

print("Per-class results:")
display(per_class_df)

print("Best model path:")
print(BEST_MODEL_PATH)

del model
del best_model
gc.collect()
torch.cuda.empty_cache()

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/football_yolo_dataset_3cls_improved/data.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=/root/.cache/huggingface/hub/models--uisikdag--yolo-v8-football-players-detection/snapshots/01c4d0e18813ac75a2c73cc35145bc240af85342/bes

,experiment,imgsz,batch,epochs,patience,time_min,precision,recall,mAP50,mAP50_95,best_model_path
0,hf_football_finetuned_3cls_improved_960,960,8,80,20,379.9,0.6836,0.6499,0.5184,0.1248,/kaggle/working/three_class_improved_experimen...


Per-class results:


,experiment,class,precision,recall,mAP50,mAP50_95
0,hf_football_finetuned_3cls_improved_960,player,0.6227,0.6667,0.4753,0.1058
1,hf_football_finetuned_3cls_improved_960,ball,0.6781,0.4948,0.4269,0.0956
2,hf_football_finetuned_3cls_improved_960,referee,0.7501,0.7881,0.6530,0.1730


Best model path:
/kaggle/working/three_class_improved_experiments/hf_football_finetuned_3cls_improved_960/weights/best.pt


# Evaluate Team A / Team B Split Accuracy

In [5]:
from ultralytics import YOLO
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from collections import defaultdict, Counter
import cv2
import numpy as np
import pandas as pd
import math

MODEL_PATH = BEST_MODEL_PATH

IOU_THRESHOLD = 0.50
CONF_THRESHOLD = 0.25
IMG_SIZE = 960

ORIGINAL_TEST_IMAGES = original_dataset_dir / "images" / "test"
ORIGINAL_TEST_LABELS = original_dataset_dir / "labels" / "test"

OUTPUT_DIR = Path("/kaggle/working/team_split_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO(MODEL_PATH)

def get_video_name(image_stem):
    if "_frame_" in image_stem:
        return image_stem.split("_frame_")[0]
    return "test_video"

def yolo_to_xyxy(xc, yc, bw, bh, img_w, img_h):
    x1 = (xc - bw / 2) * img_w
    y1 = (yc - bh / 2) * img_h
    x2 = (xc + bw / 2) * img_w
    y2 = (yc + bh / 2) * img_h
    return [float(x1), float(y1), float(x2), float(y2)]

def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)

    inter_area = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)

    union = area_a + area_b - inter_area

    if union <= 0:
        return 0.0

    return inter_area / union

def greedy_match(preds, gts, iou_threshold=0.5):
    candidates = []

    for pi, p in enumerate(preds):
        for gi, g in enumerate(gts):
            iou = box_iou(p["box"], g["box"])
            if iou >= iou_threshold:
                candidates.append((iou, pi, gi))

    candidates.sort(reverse=True)

    matched_preds = set()
    matched_gts = set()
    matches = []

    for iou, pi, gi in candidates:
        if pi in matched_preds or gi in matched_gts:
            continue

        matched_preds.add(pi)
        matched_gts.add(gi)

        matches.append({
            "pred_idx": pi,
            "gt_idx": gi,
            "iou": float(iou)
        })

    return matches

def load_original_labels(label_path, img_w, img_h):
    objects = []

    if not label_path.exists():
        return objects

    for line in label_path.read_text().splitlines():
        parts = line.strip().split()

        if len(parts) != 5:
            continue

        cls_id = int(float(parts[0]))
        xc, yc, bw, bh = map(float, parts[1:])
        box = yolo_to_xyxy(xc, yc, bw, bh, img_w, img_h)

        item = {
            "old_cls": cls_id,
            "box": box
        }

        if cls_id == 0:
            item["type"] = "player"
            item["team"] = 0

        elif cls_id == 1:
            item["type"] = "player"
            item["team"] = 1

        elif cls_id == 2:
            item["type"] = "ball"
            item["team"] = None

        elif cls_id == 3:
            item["type"] = "referee"
            item["team"] = None

        else:
            item["type"] = "unknown"
            item["team"] = None

        objects.append(item)

    return objects

def clip_box(box, w, h):
    x1, y1, x2, y2 = box
    x1 = max(0, min(int(x1), w - 1))
    y1 = max(0, min(int(y1), h - 1))
    x2 = max(0, min(int(x2), w - 1))
    y2 = max(0, min(int(y2), h - 1))
    return [x1, y1, x2, y2]

def jersey_feature(frame, box):
    h_img, w_img = frame.shape[:2]
    x1, y1, x2, y2 = clip_box(box, w_img, h_img)

    crop = frame[y1:y2, x1:x2]

    if crop.size == 0:
        return np.zeros(6, dtype=np.float32)

    h, w = crop.shape[:2]

    # Focus on upper body / shirt area
    yy1 = int(h * 0.15)
    yy2 = int(h * 0.60)
    xx1 = int(w * 0.20)
    xx2 = int(w * 0.80)

    jersey = crop[yy1:yy2, xx1:xx2]

    if jersey.size == 0:
        jersey = crop

    hsv = cv2.cvtColor(jersey, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(jersey, cv2.COLOR_BGR2LAB)

    h_channel = hsv[:, :, 0]
    s_channel = hsv[:, :, 1]
    v_channel = hsv[:, :, 2]

    green_mask = (h_channel > 35) & (h_channel < 85) & (s_channel > 40)
    dark_mask = v_channel < 30

    valid_mask = ~(green_mask | dark_mask)

    hsv_pixels = hsv[valid_mask]
    lab_pixels = lab[valid_mask]

    if len(hsv_pixels) < 20:
        hsv_pixels = hsv.reshape(-1, 3)
        lab_pixels = lab.reshape(-1, 3)

    hsv_median = np.median(hsv_pixels, axis=0) / np.array([179.0, 255.0, 255.0])
    lab_median = np.median(lab_pixels, axis=0) / 255.0

    return np.concatenate([lab_median, hsv_median]).astype(np.float32)

image_paths = sorted(ORIGINAL_TEST_IMAGES.glob("*.jpg"))

all_player_detections = []
image_level_rows = []
global_counts = Counter()

for img_path in image_paths:
    frame = cv2.imread(str(img_path))

    if frame is None:
        continue

    h, w = frame.shape[:2]
    video_name = get_video_name(img_path.stem)

    gt_objects = load_original_labels(
        ORIGINAL_TEST_LABELS / f"{img_path.stem}.txt",
        w,
        h
    )

    gt_players = [g for g in gt_objects if g["type"] == "player"]
    gt_refs = [g for g in gt_objects if g["type"] == "referee"]
    gt_balls = [g for g in gt_objects if g["type"] == "ball"]

    global_counts["gt_team_a"] += sum(1 for g in gt_players if g["team"] == 0)
    global_counts["gt_team_b"] += sum(1 for g in gt_players if g["team"] == 1)
    global_counts["gt_referee"] += len(gt_refs)
    global_counts["gt_ball"] += len(gt_balls)

    result = model.predict(
        source=str(img_path),
        imgsz=IMG_SIZE,
        conf=CONF_THRESHOLD,
        device=0,
        verbose=False
    )[0]

    pred_players = []
    pred_refs = []
    pred_balls = []

    if result.boxes is not None:
        xyxy = result.boxes.xyxy.cpu().numpy()
        clss = result.boxes.cls.cpu().numpy().astype(int)
        confs = result.boxes.conf.cpu().numpy()

        for box, cls_id, conf in zip(xyxy, clss, confs):
            item = {
                "box": [float(x) for x in box],
                "conf": float(conf),
                "cls": int(cls_id)
            }

            if cls_id == 0:
                pred_players.append(item)
            elif cls_id == 1:
                pred_balls.append(item)
            elif cls_id == 2:
                pred_refs.append(item)

    global_counts["pred_player"] += len(pred_players)
    global_counts["pred_referee"] += len(pred_refs)
    global_counts["pred_ball"] += len(pred_balls)

    player_matches = greedy_match(pred_players, gt_players, IOU_THRESHOLD)
    referee_matches = greedy_match(pred_refs, gt_refs, IOU_THRESHOLD)
    referee_as_player = greedy_match(pred_players, gt_refs, IOU_THRESHOLD)

    matched_pred_to_team = {}

    for m in player_matches:
        gt = gt_players[m["gt_idx"]]
        pred = pred_players[m["pred_idx"]]

        matched_pred_to_team[m["pred_idx"]] = gt["team"]

        if gt["team"] == 0:
            global_counts["matched_team_a"] += 1
        elif gt["team"] == 1:
            global_counts["matched_team_b"] += 1

    global_counts["matched_referee_as_referee"] += len(referee_matches)
    global_counts["referee_wrongly_as_player"] += len(referee_as_player)

    for pi, pred in enumerate(pred_players):
        feat = jersey_feature(frame, pred["box"])

        all_player_detections.append({
            "video": video_name,
            "image": img_path.name,
            "box": pred["box"],
            "conf": pred["conf"],
            "feature": feat,
            "gt_team": matched_pred_to_team.get(pi, None)
        })

    image_level_rows.append({
        "video": video_name,
        "image": img_path.name,
        "gt_team_a": sum(1 for g in gt_players if g["team"] == 0),
        "gt_team_b": sum(1 for g in gt_players if g["team"] == 1),
        "matched_team_a": sum(1 for m in player_matches if gt_players[m["gt_idx"]]["team"] == 0),
        "matched_team_b": sum(1 for m in player_matches if gt_players[m["gt_idx"]]["team"] == 1),
        "gt_referee": len(gt_refs),
        "matched_referee_as_referee": len(referee_matches),
        "referee_wrongly_as_player": len(referee_as_player),
        "pred_players": len(pred_players),
        "pred_referees": len(pred_refs),
        "pred_balls": len(pred_balls),
    })

print("Total player detections for team clustering:", len(all_player_detections))

Total player detections for team clustering: 5906


# Team Split Accuracy Summary

In [6]:
detections_by_video = defaultdict(list)

for det in all_player_detections:
    detections_by_video[det["video"]].append(det)

team_eval_rows = []
detailed_rows = []

for video_name, dets in detections_by_video.items():
    if len(dets) < 2:
        continue

    features = np.vstack([d["feature"] for d in dets])

    scaler = StandardScaler()
    X = scaler.fit_transform(features)

    kmeans = KMeans(n_clusters=2, random_state=42, n_init=20)
    clusters = kmeans.fit_predict(X)

    for d, c in zip(dets, clusters):
        d["cluster"] = int(c)

    matched = [d for d in dets if d["gt_team"] is not None]

    if len(matched) == 0:
        continue

    mappings = [
        {0: 0, 1: 1},
        {0: 1, 1: 0}
    ]

    best_mapping = None
    best_correct = -1

    for mapping in mappings:
        correct = 0

        for d in matched:
            predicted_team = mapping[d["cluster"]]

            if predicted_team == d["gt_team"]:
                correct += 1

        if correct > best_correct:
            best_correct = correct
            best_mapping = mapping

    for d in dets:
        d["pred_team"] = best_mapping[d["cluster"]]

    matched_team_a = [d for d in matched if d["gt_team"] == 0]
    matched_team_b = [d for d in matched if d["gt_team"] == 1]

    correct_team_a = sum(1 for d in matched_team_a if d["pred_team"] == d["gt_team"])
    correct_team_b = sum(1 for d in matched_team_b if d["pred_team"] == d["gt_team"])

    predicted_team_a_detections = sum(1 for d in dets if d["pred_team"] == 0)
    predicted_team_b_detections = sum(1 for d in dets if d["pred_team"] == 1)

    split_accuracy = best_correct / len(matched) if len(matched) > 0 else 0
    team_a_split_accuracy = correct_team_a / len(matched_team_a) if len(matched_team_a) > 0 else 0
    team_b_split_accuracy = correct_team_b / len(matched_team_b) if len(matched_team_b) > 0 else 0

    team_eval_rows.append({
        "video": video_name,
        "matched_players_for_split_eval": len(matched),
        "team_split_accuracy": round(split_accuracy, 4),
        "team_a_split_accuracy": round(team_a_split_accuracy, 4),
        "team_b_split_accuracy": round(team_b_split_accuracy, 4),
        "predicted_team_a_detections": predicted_team_a_detections,
        "predicted_team_b_detections": predicted_team_b_detections,
        "team_a_detection_share_after_split": round(predicted_team_a_detections / len(dets), 4),
        "team_b_detection_share_after_split": round(predicted_team_b_detections / len(dets), 4),
        "cluster_0_mapped_to": "Team A" if best_mapping[0] == 0 else "Team B",
        "cluster_1_mapped_to": "Team A" if best_mapping[1] == 0 else "Team B",
    })

    for d in dets:
        detailed_rows.append({
            "video": video_name,
            "image": d["image"],
            "conf": d["conf"],
            "cluster": d["cluster"],
            "pred_team": "Team A" if d["pred_team"] == 0 else "Team B",
            "gt_team": None if d["gt_team"] is None else ("Team A" if d["gt_team"] == 0 else "Team B"),
            "is_team_split_correct": None if d["gt_team"] is None else bool(d["pred_team"] == d["gt_team"])
        })

team_eval_df = pd.DataFrame(team_eval_rows)
detailed_df = pd.DataFrame(detailed_rows)
image_level_df = pd.DataFrame(image_level_rows)

team_a_detection_recall = global_counts["matched_team_a"] / global_counts["gt_team_a"] if global_counts["gt_team_a"] > 0 else 0
team_b_detection_recall = global_counts["matched_team_b"] / global_counts["gt_team_b"] if global_counts["gt_team_b"] > 0 else 0

referee_recall = global_counts["matched_referee_as_referee"] / global_counts["gt_referee"] if global_counts["gt_referee"] > 0 else 0
referee_wrong_as_player_rate = global_counts["referee_wrongly_as_player"] / global_counts["gt_referee"] if global_counts["gt_referee"] > 0 else 0

overall_summary = pd.DataFrame([{
    "team_a_gt_players": global_counts["gt_team_a"],
    "team_b_gt_players": global_counts["gt_team_b"],
    "team_a_detected_as_player": global_counts["matched_team_a"],
    "team_b_detected_as_player": global_counts["matched_team_b"],
    "team_a_detection_recall": round(team_a_detection_recall, 4),
    "team_b_detection_recall": round(team_b_detection_recall, 4),
    "gt_referees": global_counts["gt_referee"],
    "referees_detected_as_referee": global_counts["matched_referee_as_referee"],
    "referee_recall": round(referee_recall, 4),
    "referees_wrongly_detected_as_player": global_counts["referee_wrongly_as_player"],
    "referee_wrong_as_player_rate": round(referee_wrong_as_player_rate, 4),
    "total_predicted_players": global_counts["pred_player"],
    "total_predicted_referees": global_counts["pred_referee"],
    "total_predicted_balls": global_counts["pred_ball"],
}])

overall_summary.to_csv(OUTPUT_DIR / "overall_detection_and_referee_summary.csv", index=False)
team_eval_df.to_csv(OUTPUT_DIR / "team_split_accuracy_summary.csv", index=False)
detailed_df.to_csv(OUTPUT_DIR / "team_split_detailed_predictions.csv", index=False)
image_level_df.to_csv(OUTPUT_DIR / "image_level_detection_summary.csv", index=False)

print("Overall detection/referee summary:")
display(overall_summary)

print("Team split accuracy summary:")
display(team_eval_df)

print("Saved to:", OUTPUT_DIR)

Overall detection/referee summary:


,team_a_gt_players,team_b_gt_players,team_a_detected_as_player,team_b_detected_as_player,team_a_detection_recall,team_b_detection_recall,gt_referees,referees_detected_as_referee,referee_recall,referees_wrongly_detected_as_player,referee_wrong_as_player_rate,total_predicted_players,total_predicted_referees,total_predicted_balls
0,2408,2294,1634,1501,0.6786,0.6543,518,412,0.7954,6,0.0116,5906,594,579


Team split accuracy summary:


,video,matched_players_for_split_eval,team_split_accuracy,team_a_split_accuracy,team_b_split_accuracy,predicted_team_a_detections,predicted_team_b_detections,team_a_detection_share_after_split,team_b_detection_share_after_split,cluster_0_mapped_to,cluster_1_mapped_to
0,hashemmatch02,3135,0.9263,0.9982,0.8481,3891,2015,0.6588,0.3412,Team B,Team A


Saved to: /kaggle/working/team_split_evaluation


# Zip Training + Evaluation Results

In [7]:
import shutil
from IPython.display import FileLink, display

zip_train = shutil.make_archive(
    "/kaggle/working/three_class_improved_experiments_results",
    "zip",
    "/kaggle/working/three_class_improved_experiments"
)

zip_eval = shutil.make_archive(
    "/kaggle/working/team_split_evaluation_results",
    "zip",
    "/kaggle/working/team_split_evaluation"
)

display(FileLink(zip_train))
display(FileLink(zip_eval))
display(FileLink("/kaggle/working/three_class_improved_summary.csv"))
display(FileLink("/kaggle/working/three_class_improved_per_class.csv"))

/kaggle/working/three_class_improved_experiments_results.zip

/kaggle/working/team_split_evaluation_results.zip

/kaggle/working/three_class_improved_summary.csv

/kaggle/working/three_class_improved_per_class.csv

# Find Videos for Possession / Passes

In [8]:
from pathlib import Path

mp4_files = list(Path("/kaggle/input").rglob("*.mp4"))

print("MP4 videos found:", len(mp4_files))

for i, p in enumerate(mp4_files):
    print(i, p)

MP4 videos found: 1
0 /kaggle/input/datasets/hashemalshaweesh/hashemvid/match02 (1).mp4


In [9]:
VIDEO_PATH = "/kaggle/input/datasets/hashemalshaweesh/hashemvid/match02 (1).mp4"
MODEL_PATH = BEST_MODEL_PATH

print("Video:", VIDEO_PATH)
print("Model:", MODEL_PATH)

Video: /kaggle/input/datasets/hashemalshaweesh/hashemvid/match02 (1).mp4
Model: /kaggle/working/three_class_improved_experiments/hf_football_finetuned_3cls_improved_960/weights/best.pt


# Possession + Passes Annotated Video

In [10]:
from ultralytics import YOLO
from pathlib import Path
from collections import defaultdict, Counter, deque
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
import cv2
import numpy as np
import pandas as pd
import json
import gc
import torch

OUTPUT_DIR = Path("/kaggle/working/possession_passes_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEAM_NAMES = {
    0: "Team A",
    1: "Team B"
}

# If the colors are swapped in the output video, change this to {0: 1, 1: 0}
TEAM_CLUSTER_MAPPING = {
    0: 0,
    1: 1
}

# Manual goalkeeper override after reviewing video.
# Example: GOALKEEPER_TEAM_OVERRIDES = {14: 0, 27: 1}
GOALKEEPER_TEAM_OVERRIDES = {}

IMG_SIZE = 960
CONF_THRES = 0.25
IOU_THRES = 0.50

OUTLIER_PERCENTILE = 92
SMOOTH_WINDOW = 7
SMOOTH_MIN_COUNT = 4
LOST_MAX_FRAMES = 12
MIN_PASS_GAP_FRAMES = 12

PLAYER_CLASS = 0
BALL_CLASS = 1
REFEREE_CLASS = 2

def clip_box(box, w, h):
    x1, y1, x2, y2 = box
    x1 = max(0, min(int(x1), w - 1))
    y1 = max(0, min(int(y1), h - 1))
    x2 = max(0, min(int(x2), w - 1))
    y2 = max(0, min(int(y2), h - 1))
    return [x1, y1, x2, y2]

def box_center(box):
    x1, y1, x2, y2 = box
    return np.array([(x1 + x2) / 2, (y1 + y2) / 2], dtype=np.float32)

def foot_point(box):
    x1, y1, x2, y2 = box
    return np.array([(x1 + x2) / 2, y2], dtype=np.float32)

def is_near_goal_area(box, frame_width, margin_ratio=0.18):
    x1, y1, x2, y2 = box
    cx = (x1 + x2) / 2
    return cx <= frame_width * margin_ratio or cx >= frame_width * (1 - margin_ratio)

def jersey_feature(frame, box):
    h_img, w_img = frame.shape[:2]
    x1, y1, x2, y2 = clip_box(box, w_img, h_img)

    crop = frame[y1:y2, x1:x2]

    if crop.size == 0:
        return np.zeros(6, dtype=np.float32)

    h, w = crop.shape[:2]

    # Focus on upper body / shirt
    yy1 = int(h * 0.15)
    yy2 = int(h * 0.60)
    xx1 = int(w * 0.20)
    xx2 = int(w * 0.80)

    jersey = crop[yy1:yy2, xx1:xx2]

    if jersey.size == 0:
        jersey = crop

    hsv = cv2.cvtColor(jersey, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(jersey, cv2.COLOR_BGR2LAB)

    h_channel = hsv[:, :, 0]
    s_channel = hsv[:, :, 1]
    v_channel = hsv[:, :, 2]

    green_mask = (h_channel > 35) & (h_channel < 85) & (s_channel > 40)
    dark_mask = v_channel < 30

    valid_mask = ~(green_mask | dark_mask)

    lab_pixels = lab[valid_mask]
    hsv_pixels = hsv[valid_mask]

    if len(lab_pixels) < 20:
        lab_pixels = lab.reshape(-1, 3)
        hsv_pixels = hsv.reshape(-1, 3)

    lab_median = np.median(lab_pixels, axis=0) / 255.0
    hsv_median = np.median(hsv_pixels, axis=0) / np.array([179.0, 255.0, 255.0])

    return np.concatenate([lab_median, hsv_median]).astype(np.float32)

def choose_ball(ball_dets):
    if len(ball_dets) == 0:
        return None
    return max(ball_dets, key=lambda d: d["conf"])

def assign_team_with_goalkeeper_logic(players, frame_width):
    known_players = []

    for p in players:
        if p.get("is_color_outlier", False):
            continue

        if p.get("team") is not None:
            known_players.append(p)

    team_centroids = {}

    for team_id in [0, 1]:
        pts = [box_center(p["box"]) for p in known_players if p.get("team") == team_id]

        if len(pts) > 0:
            team_centroids[team_id] = np.mean(np.vstack(pts), axis=0)

    filtered_players = []

    for p in players:
        tid = p.get("track_id", -1)

        if tid in GOALKEEPER_TEAM_OVERRIDES:
            p["team"] = int(GOALKEEPER_TEAM_OVERRIDES[tid])
            p["role"] = "goalkeeper_manual"
            p["assigned_by"] = "manual_override"
            filtered_players.append(p)
            continue

        is_outlier = p.get("is_color_outlier", False)
        near_goal = is_near_goal_area(p["box"], frame_width)

        # Normal player
        if not is_outlier and p.get("team") is not None:
            p["role"] = "player"
            filtered_players.append(p)
            continue

        # Goalkeeper candidate: different color but close to goal area
        if is_outlier and near_goal:
            p_center = box_center(p["box"])

            if len(team_centroids) == 2:
                d0 = np.linalg.norm(p_center - team_centroids[0])
                d1 = np.linalg.norm(p_center - team_centroids[1])
                p["team"] = 0 if d0 <= d1 else 1
                p["role"] = "goalkeeper_candidate"
                p["assigned_by"] = "goalkeeper_spatial_fallback"
                filtered_players.append(p)

            continue

        # Outlier not near goal is likely referee or unknown false player
        p["role"] = "ignored_outlier"
        p["team"] = None

    return filtered_players

def find_owner(players, ball_box, frame_width):
    if ball_box is None or len(players) == 0:
        return None

    ball_c = box_center(ball_box)

    heights = [p["box"][3] - p["box"][1] for p in players]
    median_h = np.median(heights) if len(heights) > 0 else frame_width * 0.08

    max_dist = max(45, min(160, median_h * 1.25))

    best = None
    best_dist = 10**9

    for p in players:
        if p.get("track_id", -1) < 0:
            continue

        if p.get("team") is None:
            continue

        foot = foot_point(p["box"])
        center = box_center(p["box"])

        dist_foot = np.linalg.norm(ball_c - foot)
        dist_center = np.linalg.norm(ball_c - center)

        dist = min(dist_foot, dist_center * 0.75)

        if dist < best_dist:
            best_dist = dist
            best = p

    if best is not None and best_dist <= max_dist:
        return {
            "track_id": int(best["track_id"]),
            "team": int(best["team"]),
            "distance": float(best_dist),
            "threshold": float(max_dist)
        }

    return None

def draw_label(frame, text, x, y, color):
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.5
    thickness = 1

    (tw, th), _ = cv2.getTextSize(text, font, scale, thickness)
    x2 = x + tw + 8
    y2 = y

    cv2.rectangle(frame, (x, y - th - 8), (x2, y2), color, -1)
    cv2.putText(frame, text, (x + 4, y - 5), font, scale, (255, 255, 255), thickness)

def draw_scoreboard(frame, team0_pct, team1_pct, passes, current_owner_text):
    h, w = frame.shape[:2]

    panel_x1, panel_y1 = 10, 10
    panel_x2, panel_y2 = min(w - 10, 920), 120

    cv2.rectangle(frame, (panel_x1, panel_y1), (panel_x2, panel_y2), (0, 0, 0), -1)
    cv2.rectangle(frame, (panel_x1, panel_y1), (panel_x2, panel_y2), (255, 255, 255), 2)

    line1 = f"Possession | Team A: {team0_pct:.1f}% | Team B: {team1_pct:.1f}%"
    line2 = f"Passes     | Team A: {passes[0]} | Team B: {passes[1]}"
    line3 = f"Current Owner: {current_owner_text}"

    cv2.putText(frame, line1, (25, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
    cv2.putText(frame, line2, (25, 78), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
    cv2.putText(frame, line3, (25, 108), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

model = YOLO(MODEL_PATH)

print("Model classes:", model.names)
print("Running detection + tracking...")

raw_frames = defaultdict(list)
feature_records = []

results = model.track(
    source=VIDEO_PATH,
    stream=True,
    imgsz=IMG_SIZE,
    conf=CONF_THRES,
    iou=IOU_THRES,
    device=0,
    tracker="bytetrack.yaml",
    persist=True,
    verbose=False,
    save=False
)

for frame_idx, result in enumerate(tqdm(results)):
    frame = result.orig_img
    h_img, w_img = frame.shape[:2]

    if result.boxes is None:
        continue

    boxes = result.boxes

    xyxy = boxes.xyxy.cpu().numpy() if boxes.xyxy is not None else []
    clss = boxes.cls.cpu().numpy().astype(int) if boxes.cls is not None else []
    confs = boxes.conf.cpu().numpy() if boxes.conf is not None else []

    if boxes.id is not None:
        ids = boxes.id.cpu().numpy().astype(int)
    else:
        ids = np.array([-1] * len(xyxy))

    for box, cls_id, conf, track_id in zip(xyxy, clss, confs, ids):
        box = clip_box(box, w_img, h_img)

        if cls_id == PLAYER_CLASS:
            feat = jersey_feature(frame, box)
            feature_idx = len(feature_records)

            feature_records.append({
                "feature": feat,
                "track_id": int(track_id),
                "frame_idx": int(frame_idx)
            })

            raw_frames[frame_idx].append({
                "cls": "player",
                "box": box,
                "conf": float(conf),
                "track_id": int(track_id),
                "feature_idx": feature_idx
            })

        elif cls_id == BALL_CLASS:
            raw_frames[frame_idx].append({
                "cls": "ball",
                "box": box,
                "conf": float(conf),
                "track_id": int(track_id)
            })

        elif cls_id == REFEREE_CLASS:
            raw_frames[frame_idx].append({
                "cls": "referee",
                "box": box,
                "conf": float(conf),
                "track_id": int(track_id)
            })

print("Frames with detections:", len(raw_frames))
print("Player color samples:", len(feature_records))

if len(feature_records) < 20:
    raise ValueError("Not enough player detections for team clustering.")

# Team clustering
features = np.vstack([r["feature"] for r in feature_records])

scaler = StandardScaler()
X = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=2, random_state=42, n_init=20)
clusters = kmeans.fit_predict(X)
distances = np.min(kmeans.transform(X), axis=1)

outlier_threshold = np.percentile(distances, OUTLIER_PERCENTILE)

for i, rec in enumerate(feature_records):
    rec["cluster"] = int(clusters[i])
    rec["distance"] = float(distances[i])
    rec["is_outlier"] = bool(distances[i] > outlier_threshold)

for frame_idx, dets in raw_frames.items():
    for det in dets:
        if det["cls"] != "player":
            continue

        rec = feature_records[det["feature_idx"]]

        det["cluster"] = rec["cluster"]
        det["team"] = TEAM_CLUSTER_MAPPING[int(rec["cluster"])]
        det["color_distance"] = rec["distance"]
        det["is_color_outlier"] = rec["is_outlier"]

# Majority vote per track ID
track_votes = defaultdict(Counter)

for frame_idx, dets in raw_frames.items():
    for det in dets:
        if det["cls"] != "player":
            continue

        tid = det["track_id"]

        if tid < 0:
            continue

        if det.get("is_color_outlier", False):
            continue

        track_votes[tid][det["team"]] += 1

track_team = {}

for tid, votes in track_votes.items():
    if sum(votes.values()) >= 3:
        track_team[tid] = votes.most_common(1)[0][0]

print("Tracked players with team assignment:", len(track_team))
print("Outlier threshold:", outlier_threshold)

# Video reader/writer
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 25

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

output_video_path = OUTPUT_DIR / "annotated_possession_passes.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(output_video_path), fourcc, fps, (width, height))

recent_owners = deque(maxlen=SMOOTH_WINDOW)

possession_frames = Counter()
passes = Counter()

last_seen_owner = None
last_seen_frame = -10**9

last_counted_owner = None
last_owner_change_frame = -10**9

frame_rows = []

team_colors = {
    0: (40, 180, 40),
    1: (220, 80, 40)
}

ball_color = (40, 220, 255)
referee_color = (180, 60, 220)

print("Calculating possession and passes...")

frame_idx = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    dets = raw_frames.get(frame_idx, [])

    players = []
    balls = []
    referees = []

    for det in dets:
        if det["cls"] == "player":
            p = det.copy()
            tid = p.get("track_id", -1)

            if tid in track_team and not p.get("is_color_outlier", False):
                p["team"] = int(track_team[tid])
                p["assigned_by"] = "track_majority"

            elif not p.get("is_color_outlier", False):
                p["team"] = int(p["team"])
                p["assigned_by"] = "frame_cluster"

            else:
                p["team"] = None
                p["assigned_by"] = "needs_goalkeeper_logic"

            players.append(p)

        elif det["cls"] == "ball":
            balls.append(det)

        elif det["cls"] == "referee":
            referees.append(det)

    players = assign_team_with_goalkeeper_logic(players, width)

    ball_det = choose_ball(balls)
    ball_box = ball_det["box"] if ball_det is not None else None

    owner = find_owner(players, ball_box, width)

    if owner is not None:
        recent_owners.append((owner["track_id"], owner["team"]))
    else:
        recent_owners.append(None)

    valid_owners = [x for x in recent_owners if x is not None]

    current_owner = None

    if len(valid_owners) > 0:
        owner_counts = Counter(valid_owners)
        most_common_owner, count = owner_counts.most_common(1)[0]

        if count >= SMOOTH_MIN_COUNT:
            current_owner = most_common_owner

    if current_owner is None:
        if last_seen_owner is not None and (frame_idx - last_seen_frame) <= LOST_MAX_FRAMES:
            current_owner = last_seen_owner

    if current_owner is not None:
        owner_id, owner_team = current_owner

        possession_frames[owner_team] += 1

        if last_counted_owner is not None:
            previous_id, previous_team = last_counted_owner

            if owner_id != previous_id:
                if owner_team == previous_team:
                    if (frame_idx - last_owner_change_frame) >= MIN_PASS_GAP_FRAMES:
                        passes[owner_team] += 1

                last_owner_change_frame = frame_idx

        last_counted_owner = current_owner
        last_seen_owner = current_owner
        last_seen_frame = frame_idx

    # Draw players
    for p in players:
        x1, y1, x2, y2 = p["box"]
        team = p.get("team", None)

        if team is None:
            continue

        color = team_colors.get(team, (180, 180, 180))

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        label = f"{TEAM_NAMES[team]} ID {p.get('track_id', -1)}"

        if p.get("role") in ["goalkeeper_candidate", "goalkeeper_manual"]:
            label += " GK"

        draw_label(frame, label, x1, max(25, y1), color)

    # Draw ball
    if ball_box is not None:
        x1, y1, x2, y2 = ball_box
        cv2.rectangle(frame, (x1, y1), (x2, y2), ball_color, 2)
        draw_label(frame, "ball", x1, max(25, y1), ball_color)

    # Draw referees
    for r in referees:
        x1, y1, x2, y2 = r["box"]
        cv2.rectangle(frame, (x1, y1), (x2, y2), referee_color, 2)
        draw_label(frame, "referee", x1, max(25, y1), referee_color)

    total_pos_frames = sum(possession_frames.values())

    if total_pos_frames > 0:
        team0_pct = 100 * possession_frames[0] / total_pos_frames
        team1_pct = 100 * possession_frames[1] / total_pos_frames
    else:
        team0_pct = 0
        team1_pct = 0

    if current_owner is not None:
        current_owner_text = f"{TEAM_NAMES[current_owner[1]]} - ID {current_owner[0]}"
    else:
        current_owner_text = "None"

    draw_scoreboard(frame, team0_pct, team1_pct, passes, current_owner_text)

    owner_id = None
    owner_team_name = None

    if current_owner is not None:
        owner_id = int(current_owner[0])
        owner_team_name = TEAM_NAMES[int(current_owner[1])]

    frame_rows.append({
        "frame": frame_idx,
        "time_sec": frame_idx / fps,
        "owner_track_id": owner_id,
        "owner_team": owner_team_name,
        "team_a_possession_frames": possession_frames[0],
        "team_b_possession_frames": possession_frames[1],
        "team_a_passes": passes[0],
        "team_b_passes": passes[1],
        "players_used": len(players),
        "referees_detected": len(referees),
        "balls_detected": len(balls),
    })

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

total_pos_frames = sum(possession_frames.values())

summary_rows = []

for team_id in [0, 1]:
    pct = 100 * possession_frames[team_id] / total_pos_frames if total_pos_frames > 0 else 0

    summary_rows.append({
        "team": TEAM_NAMES[team_id],
        "possession_frames": int(possession_frames[team_id]),
        "possession_seconds": round(possession_frames[team_id] / fps, 2),
        "possession_percentage": round(pct, 2),
        "passes": int(passes[team_id])
    })

summary_df = pd.DataFrame(summary_rows)
frames_df = pd.DataFrame(frame_rows)

summary_path = OUTPUT_DIR / "possession_passes_summary.csv"
frames_path = OUTPUT_DIR / "frame_level_possession.csv"
track_team_path = OUTPUT_DIR / "track_team_assignments.json"

summary_df.to_csv(summary_path, index=False)
frames_df.to_csv(frames_path, index=False)

with open(track_team_path, "w") as f:
    json.dump({str(k): int(v) for k, v in track_team.items()}, f, indent=2)

print("Done.")
print("Annotated video:", output_video_path)
print("Summary:", summary_path)
print("Frame-level CSV:", frames_path)

display(summary_df)

gc.collect()
torch.cuda.empty_cache()

Model classes: {0: 'player', 1: 'ball', 2: 'referee'}
Running detection + tracking...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 187ms
Prepared 1 package in 30ms
Installed 1 package in 5ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



0it [00:00, ?it/s]

Frames with detections: 3597
Player color samples: 27024
Tracked players with team assignment: 354
Outlier threshold: 3.0888789
Calculating possession and passes...
Done.
Annotated video: /kaggle/working/possession_passes_output/annotated_possession_passes.mp4
Summary: /kaggle/working/possession_passes_output/possession_passes_summary.csv
Frame-level CSV: /kaggle/working/possession_passes_output/frame_level_possession.csv


,team,possession_frames,possession_seconds,possession_percentage,passes
0,Team A,2717,90.66,75.77,29
1,Team B,869,29.00,24.23,1


# Download Final Possession Outputs

In [11]:
import shutil
from IPython.display import FileLink, display

zip_path = shutil.make_archive(
    "/kaggle/working/possession_passes_output",
    "zip",
    "/kaggle/working/possession_passes_output"
)

display(FileLink(zip_path))
display(FileLink("/kaggle/working/possession_passes_output/possession_passes_summary.csv"))

/kaggle/working/possession_passes_output.zip

/kaggle/working/possession_passes_output/possession_passes_summary.csv